# 01 - Plain text NER (Task 1) and LLM chat (Task 4)

Pipeline: `validation.jsonl` -> GLiNER -> predictions (`results/`) -> strict / relaxed evaluation.

- Task 1: `gliner_small-v2.5`, `gliner_small-v2.5` + label descriptions, `gliner_large-v2.5`.
- Task 4: zero-shot NER with a chat LLM (no API) on `validation_small.jsonl`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import (
    LABEL_DESCRIPTIONS,
    get_dataset_labels,
    load_gliner_model,
    predict_dataset,
)
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = ROOT / "data" / "plain" / "validation.jsonl"
VALIDATION_SMALL_PATH = ROOT / "data" / "plain" / "validation_small.jsonl"
RESULTS_PLAIN = ROOT / "results" / "plain"
RESULTS_LLM = ROOT / "results" / "llm"
THRESHOLD = 0.5

In [2]:
gold = load_jsonl(VALIDATION_PATH)
assert gold, f"{VALIDATION_PATH} is empty: copy the course data/ folder into the project first."
LABELS = get_dataset_labels(gold)
print(f"{len(gold)} gold documents | labels: {LABELS}")

50 gold documents | labels: ['ADDRESS', 'AMOUNT', 'DATE', 'ID_CARD_NUMBER', 'ORGANIZATION', 'PERSON_NAME', 'REGISTRATION_NUMBER']


## What are the `spans` returned by `predict_entities`?

Each predicted entity is a dict with:

- `start`: index of the first character of the mention (0-based, inclusive).
- `end`: index after the last character (exclusive), so `text[start:end] == entity["text"]`.
- `text`: the mention extracted from the document.
- `label`: the predicted entity type.
- `score`: model confidence (0-1); entities below `threshold` are discarded.

Strict evaluation needs an exact match of `(start, end, label)`.

In [3]:
model_small = load_gliner_model("gliner-community/gliner_small-v2.5")

sample = gold[0]
for ent in model_small.predict_entities(sample["text"], LABELS, threshold=THRESHOLD)[:8]:
    assert sample["text"][ent["start"]:ent["end"]] == ent["text"]
    print(ent["start"], ent["end"], ent["label"], round(ent["score"], 2), repr(ent["text"]))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

0 35 ORGANIZATION 0.93 'Servicio de Procedimientos Públicos'
42 56 ORGANIZATION 0.98 'Región Central'
72 98 PERSON_NAME 0.98 'Patricia Giménez Domínguez'
127 136 ID_CARD_NUMBER 0.76 '12082063W'
161 178 ADDRESS 0.97 'Avenida de Aragón'
180 188 ADDRESS 0.7 '102, 2-B'
190 202 ADDRESS 0.83 '17001 Girona'
216 226 DATE 0.97 '2018-04-09'


## Task 1.1 - `gliner_small-v2.5` (no label descriptions)

In [4]:
SMALL_PATH = RESULTS_PLAIN / "gliner_small_validation.jsonl"

preds = predict_dataset(model_small, gold, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, SMALL_PATH)

strict_small = evaluate_ner_strict(VALIDATION_PATH, SMALL_PATH)
relaxed_small = evaluate_ner_relaxed(VALIDATION_PATH, SMALL_PATH)

Running GLiNER inference: 100%|██████████| 50/50 [01:34<00:00,  1.89s/it]


STRICT EVALUATION
Micro -> P: 0.77 | R: 0.87 | F1: 0.82 | TP: 1787 | FP: 544 | FN: 263

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00     499       0       1
ID_CARD_NUMBER              0.96   0.99   0.98     198       8       2
ADDRESS                     0.00   0.00   0.00       0     434     200
REGISTRATION_NUMBER         0.99   0.98   0.98     245       3       5
DATE                        1.00   1.00   1.00     399       0       1
AMOUNT                      0.95   0.93   0.94     186       9      14
ORGANIZATION                0.74   0.87   0.80     260      90      40

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.77 | R: 0.87 | F1: 0.82 | TP: 1787 | FP: 544 | FN: 263

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------------

## Task 1.2 - `gliner_small-v2.5` + label descriptions

The model receives a natural-language description of each label instead of the bare label name;
the original label name is restored in the output (see `ner.inference.predict_entities`).

In [5]:
SMALL_DESC_PATH = RESULTS_PLAIN / "gliner_small_descriptions_validation.jsonl"
descriptions = {lbl: LABEL_DESCRIPTIONS.get(lbl, lbl) for lbl in LABELS}

preds = predict_dataset(model_small, gold, labels=descriptions, threshold=THRESHOLD)
save_responses_to_jsonl(preds, SMALL_DESC_PATH)

strict_small_desc = evaluate_ner_strict(VALIDATION_PATH, SMALL_DESC_PATH)
relaxed_small_desc = evaluate_ner_relaxed(VALIDATION_PATH, SMALL_DESC_PATH)

Running GLiNER inference:  42%|████▏     | 21/50 [00:59<01:21,  2.82s/it]


KeyboardInterrupt: 

## Task 1.3 - `gliner_large-v2.5`

In [6]:
import gc

del model_small
gc.collect()

LARGE_PATH = RESULTS_PLAIN / "gliner_large_validation.jsonl"
model_large = load_gliner_model("gliner-community/gliner_large-v2.5")

preds = predict_dataset(model_large, gold, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, LARGE_PATH)

strict_large = evaluate_ner_strict(VALIDATION_PATH, LARGE_PATH)
relaxed_large = evaluate_ner_relaxed(VALIDATION_PATH, LARGE_PATH)

del model_large
gc.collect()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Running GLiNER inference: 100%|██████████| 50/50 [05:54<00:00,  7.10s/it]



STRICT EVALUATION
Micro -> P: 1.00 | R: 0.91 | F1: 0.96 | TP: 1875 | FP: 0 | FN: 175

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00     500       0       0
ID_CARD_NUMBER              1.00   0.97   0.99     195       0       5
ADDRESS                     1.00   0.94   0.97     188       0      12
REGISTRATION_NUMBER         1.00   0.37   0.54      93       0     157
DATE                        1.00   1.00   1.00     399       0       1
AMOUNT                      1.00   1.00   1.00     200       0       0
ORGANIZATION                1.00   1.00   1.00     300       0       0

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 0.91 | F1: 0.96 | TP: 1875 | FP: 0 | FN: 175

ENTITY TYPE                    P      R     F1      TP      FP      FN
-------------------------------------------------------------------------------

0

## Task 4 - NER with a chat LLM (no API)

1. Run the next cell: it writes `results/llm/prompt.txt`.
2. Paste the prompt in the chat of an LLM (ChatGPT, Gemini, Claude...).
3. Save the answer (JSONL) in `results/llm/raw_llm_response.txt`.
4. Run the cell after that: it parses the answer, recomputes `start`/`end` on the original text
   (LLMs cannot count characters) and evaluates against `validation_small.jsonl`.

In [7]:
from ner.llm import build_llm_prompt

gold_small = load_jsonl(VALIDATION_SMALL_PATH)
assert gold_small, f"{VALIDATION_SMALL_PATH} is empty"

RESULTS_LLM.mkdir(parents=True, exist_ok=True)
prompt = build_llm_prompt(gold_small, descriptions)
(RESULTS_LLM / "prompt.txt").write_text(prompt, encoding="utf-8")
print(f"{len(gold_small)} documents -> {RESULTS_LLM / 'prompt.txt'} ({len(prompt)} characters)")

10 documents -> c:\Users\riset\Personal\Proyectos\DAM2\InteligenciaArtificial\IA\results\llm\prompt.txt (38830 characters)


In [8]:
from ner.llm import llm_records_to_predictions, parse_llm_response

RAW_PATH = RESULTS_LLM / "raw_llm_response.txt"
LLM_PATH = RESULTS_LLM / "llm_validation.jsonl"

parsed = parse_llm_response(RAW_PATH.read_text(encoding="utf-8"))
llm_preds = llm_records_to_predictions(parsed, gold_small, valid_labels=LABELS)
save_responses_to_jsonl(llm_preds, LLM_PATH)

n_raw = sum(len(r.get("entities", [])) for r in parsed)
n_kept = sum(len(p["entities"]) for p in llm_preds)
print(f"{len(parsed)}/{len(gold_small)} documents answered | entities kept after alignment: {n_kept}/{n_raw}")

strict_llm = evaluate_ner_strict(VALIDATION_SMALL_PATH, LLM_PATH)
relaxed_llm = evaluate_ner_relaxed(VALIDATION_SMALL_PATH, LLM_PATH)

10/10 documents answered | entities kept after alignment: 403/403

STRICT EVALUATION
Micro -> P: 0.99 | R: 0.97 | F1: 0.98 | TP: 399 | FP: 4 | FN: 11

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00     100       0       0
ID_CARD_NUMBER              1.00   0.90   0.95      36       0       4
ADDRESS                     1.00   1.00   1.00      40       0       0
REGISTRATION_NUMBER         1.00   0.94   0.97      47       0       3
DATE                        1.00   1.00   1.00      80       0       0
AMOUNT                      1.00   1.00   1.00      40       0       0
ORGANIZATION                0.93   0.93   0.93      56       4       4

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.99 | R: 0.97 | F1: 0.98 | TP: 399 | FP: 4 | FN: 11

ENTITY TYPE                    P      R     F1      TP      FP      FN
-----------------

## Results summary (all experiments found in `results/`)

In [9]:
from ner.report import build_results_table

table, missing = build_results_table(ROOT)
print(table)
if missing:
    print("\nNot evaluated yet:", *missing, sep="\n  - ")

| Input | Model | Label descriptions | Extraction | Evaluation | P | R | F1 |
| :--- | :--- | :---: | :---: | :--- | :---: | :---: | :---: |
| Plain text | `gliner_small-v2.5` | No | — | Strict | 0.77 | 0.87 | 0.82 |
| Plain text | `gliner_small-v2.5` | No | — | Relaxed occurrences | 0.77 | 0.87 | 0.82 |
| Plain text | `gliner_small-v2.5` | No | — | Relaxed unique | 0.78 | 0.89 | 0.83 |
| Plain text | `gliner_small-v2.5` | Yes | — | Strict | 0.98 | 0.28 | 0.43 |
| Plain text | `gliner_small-v2.5` | Yes | — | Relaxed occurrences | 0.98 | 0.28 | 0.43 |
| Plain text | `gliner_small-v2.5` | Yes | — | Relaxed unique | 0.98 | 0.43 | 0.59 |
| Plain text | `gliner_large-v2.5` | No | — | Strict | 1.00 | 0.91 | 0.96 |
| Plain text | `gliner_large-v2.5` | No | — | Relaxed occurrences | 1.00 | 0.91 | 0.96 |
| Plain text | `gliner_large-v2.5` | No | — | Relaxed unique | 1.00 | 0.91 | 0.95 |
| Clean PDF | `gliner_small-v2.5` | No | PyMuPDF | Strict | 0.53 | 0.60 | 0.56 |
| Clean PDF | `gliner_small-